In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faiss-cpu"])

import numpy as np
import pandas as pd
import time

df = pd.read_parquet("laptop_chunks_embeddings_with_lineage.parquet")
print(df.shape)

# embeddings come out of parquet as object/list columns -- stack into a proper float32 matrix
embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
print("embedding matrix:", embeddings.shape, embeddings.dtype)


In [ ]:
# 30+ queries with a known-relevant chunk_id per spec requirement
# Build this by picking real chunks from your data and writing a query that should retrieve them
import random

random.seed(42)
sample_idx = random.sample(range(len(df)), 35)

eval_queries = []
for idx in sample_idx:
    row = df.iloc[idx]
    # naive auto-generated query from title+cpu — replace with better hand-written queries where you can
    query_text = f"{row['title']} laptop with {row['cpu']}"
    eval_queries.append({
        "query": query_text,
        "relevant_chunk_id": row["chunk_id"],
        "relevant_row_uid": row["row_uid"],
    })

eval_df = pd.DataFrame(eval_queries)
eval_df.to_csv("eval_queries.csv", index=False)
print(eval_df.shape)
eval_df.head()

(35, 3)


,query,relevant_chunk_id,relevant_row_uid
0,HP 250 G5 laptop with Intel Pentium Quad Core ...,31c2346ceb3c05e9809dbc6dd0c92b85_0,31c2346ceb3c05e9809dbc6dd0c92b85
1,ASUS Vivobook 17 Laptop - 17.3 FHD Display - I...,2107f685d84ec0a93525ba7e9ac0e838_1,2107f685d84ec0a93525ba7e9ac0e838
2,"HP 17.3 FHD i5 Laptop,Intel Core i5-1335U,WiFi...",84e62cbf3903024ae9ccf355dcb1e66b_1,84e62cbf3903024ae9ccf355dcb1e66b
3,"HP 15.6"" Full HD LED Notebook,AMD Ryzen 5 7520...",d0476446e9aa8e6dfe8d614bc27804bb_1,d0476446e9aa8e6dfe8d614bc27804bb
4,Dell Latitude 7430 Laptop Intel i7-1265U 32GB ...,a116430abaaeb6b2127335fef2e104e5_2,a116430abaaeb6b2127335fef2e104e5


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # must match embedding model used in M2
query_embeddings = model.encode(eval_df["query"].tolist(), batch_size=32, show_progress_bar=False).astype("float32")
print(query_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(35, 384)


In [ ]:
import faiss

d = embeddings.shape[1]  # 384
n = embeddings.shape[0]

def build_and_time(build_fn, name):
    t0 = time.perf_counter()
    index = build_fn()
    build_time_s = time.perf_counter() - t0
    mem_mb = len(faiss.serialize_index(index)) / (1024 ** 2)  # serialized size = actual index memory footprint
    print(f"{name:20s} | build: {build_time_s:.3f}s | memory: {mem_mb:.2f} MB")
    return index, build_time_s, mem_mb

# 1. Baseline: exact brute-force search
def _build_flat():
    idx = faiss.IndexFlatL2(d)
    idx.add(embeddings)
    return idx

# 2. IVF: partitions the space into clusters, searches only nearby clusters
def _build_ivf():
    nlist = min(50, n // 10)  # rule of thumb: sqrt(n) to n/40; keep small since n is small
    quantizer = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFFlat(quantizer, d, nlist)
    idx.train(embeddings)   # training time counts as part of "build" for IVF, so it's included here
    idx.add(embeddings)
    idx.nprobe = 8  # how many clusters to search at query time -- tune for recall/speed tradeoff
    return idx

# 3. HNSW: graph-based ANN, generally strong recall/latency tradeoff, no training needed
def _build_hnsw():
    idx = faiss.IndexHNSWFlat(d, 32)  # 32 = M, number of neighbor links per node
    idx.hnsw.efConstruction = 40
    idx.add(embeddings)
    idx.hnsw.efSearch = 32
    return idx

index_flat, flat_build_s, flat_mem_mb = build_and_time(_build_flat, "IndexFlatL2 (baseline)")
index_ivf, ivf_build_s, ivf_mem_mb = build_and_time(_build_ivf, "IndexIVFFlat")
index_hnsw, hnsw_build_s, hnsw_mem_mb = build_and_time(_build_hnsw, "IndexHNSWFlat")

print("IndexFlatL2 ntotal:", index_flat.ntotal)
print("IndexIVFFlat ntotal:", index_ivf.ntotal)
print("IndexHNSWFlat ntotal:", index_hnsw.ntotal)

# keyed by the exact name strings evaluate_index() will use below, so they merge cleanly in cell 4
build_stats = {
    "IndexFlatL2 (baseline)": (flat_build_s, flat_mem_mb),
    "IndexIVFFlat": (ivf_build_s, ivf_mem_mb),
    "IndexHNSWFlat": (hnsw_build_s, hnsw_mem_mb),
}


In [ ]:
def evaluate_index(index, name, query_vecs, k=5):
    # ground truth: exact search always defines "correct" neighbors
    _, gt_indices = index_flat.search(query_vecs, k)

    start = time.perf_counter()
    _, pred_indices = index.search(query_vecs, k)
    elapsed = time.perf_counter() - start
    latency_ms_per_query = (elapsed / len(query_vecs)) * 1000

    # recall@k: fraction of ground-truth neighbors that show up in this index's top-k
    hits = 0
    total = 0
    for gt_row, pred_row in zip(gt_indices, pred_indices):
        hits += len(set(gt_row.tolist()) & set(pred_row.tolist()))
        total += len(gt_row)
    recall_at_k = hits / total

    build_time_s, memory_mb = build_stats[name]
    print(f"{name:20s} | recall@{k}: {recall_at_k:.3f} | avg latency: {latency_ms_per_query:.3f} ms/query "
          f"| build: {build_time_s:.3f}s | memory: {memory_mb:.2f} MB")
    return {
        "index": name,
        "recall_at_k": recall_at_k,
        "latency_ms": latency_ms_per_query,
        "build_time_s": build_time_s,
        "memory_mb": memory_mb,
    }

results = []
results.append(evaluate_index(index_flat, "IndexFlatL2 (baseline)", query_embeddings, k=5))
results.append(evaluate_index(index_ivf, "IndexIVFFlat", query_embeddings, k=5))
results.append(evaluate_index(index_hnsw, "IndexHNSWFlat", query_embeddings, k=5))

results_df = pd.DataFrame(results)
results_df.to_csv("m3_index_comparison.csv", index=False)
results_df


In [ ]:
def evaluate_against_labels(index, name, k=5):
    D, I = index.search(query_embeddings, k)
    hits = 0
    for i, row in eval_df.iterrows():
        retrieved_uids = df.iloc[I[i]]["row_uid"].tolist()
        if row["relevant_row_uid"] in retrieved_uids:
            hits += 1
    print(f"{name:20s} | hit rate @ {k} (found true relevant device): {hits}/{len(eval_df)} = {hits/len(eval_df):.3f}")
    return hits / len(eval_df)

for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat")]:
    evaluate_against_labels(idx, name)

IndexFlatL2          | hit rate @ 5 (found true relevant device): 23/35 = 0.657
IndexIVFFlat         | hit rate @ 5 (found true relevant device): 19/35 = 0.543
IndexHNSWFlat        | hit rate @ 5 (found true relevant device): 22/35 = 0.629


In [ ]:
# pick whichever wins your recall/latency tradeoff — HNSW is a common good default at this scale
faiss.write_index(index_hnsw, "laptop_index_hnsw.faiss")

# also save the row lookup so FAISS integer positions map back to your data
df[["chunk_id", "row_uid", "title", "chunk_text"]].to_csv("faiss_id_lookup.csv", index=False)

from google.colab import files
files.download("laptop_index_hnsw.faiss")
files.download("faiss_id_lookup.csv")
files.download("m3_index_comparison.csv")
files.download("eval_queries.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>